# 5.11 同一段文字出現很多次會怎樣？


資料裡同一句重複一百次，模型就會遇到一百次類似更新，這不等於一百份新知識。重複還可能讓同一句跨到訓練與驗證兩邊，重現 [1.2](https://birdhackor.github.io/tiny-perceptron-vlm/1.2.html) 的洩漏。但兩個看起來一樣的句子，可能只是多一個空格，直接比原文就漏掉它。我們先明示哪些差異可以忽略，再找完全重複。

正規化（normalization）是在比較前統一某些寫法，這裡只合並空白：`"貓  看狗"` 與 `"貓 看狗"` 都變成 `"貓 看狗"`。這不是 [4.3](https://birdhackor.github.io/tiny-perceptron-vlm/4.3.html) 的數值正規化，雖然名稱相同，處理對象與規則不同。我們沒有刪掉所有空格，也沒有改字義或標點。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
from tiny_perceptron.data import fingerprint

docs = ["貓  看狗", "貓 看狗", "狗 看貓"]
normalized = [" ".join(text.split()) for text in docs]
hashes = [fingerprint(text) for text in docs]
print(normalized)
print("原始筆數", len(docs), "正規化後不同內容", len(set(hashes)))
print("指紋字元數", len(hashes[0]))
assert hashes[0] == hashes[1]

輸入三段文字，前兩段只有空白數量不同。`split()` 不給參數時按空白分段，`" ".join(...)` 用單一空格接回，第一行應顯示兩份 `貓 看狗` 與一份 `狗 看貓`。`fingerprint` 使用同樣規則再算SHA-256，輸出固定長的內容指紋，也叫hash。內容相同得到同一指紋，所以三個原始記錄只剩兩種不同內容；完整指紋有64個十六進位字元，也就是每格使用0–9或a–f這16種符號之一。

我們常用指紋比較長文件，不必每次逐字掃描。它不是可還原的文字壓縮，也不證明兩段文字在意思上相同；換一個字往往會讓指紋整體改變。碰撞指兩份不同的正規化內容偶然得到同一個完整指紋；SHA-256發生這種情況的機會極小，但判定重要重複時仍可保留原文核對。只取極短前綴則丟掉了許多區分資訊，更容易讓不同內容對上相同短字串，不能當成絕對唯一。

正規化規則必須按資料種類選擇。程式碼的縮排可能有意義，詩的換行與連續空白可能有排版功能，字串裡面的空白也可能屬於值。把所有資料套進這一份規則，會把原本不同的內容誤當重複。應先抽查樣本，記下規則，再去重與分組，而不是單看去掉多少比例。

重複也會改訓練分配：某個來源複製更多次，會更常被抽到，像 [1.4](https://birdhackor.github.io/tiny-perceptron-vlm/1.4.html) 的字頻會因重複增加。因此即使重複只留在訓練組，仍影響我們究竟強調什麼。去重是在控制統計與切分，不自動保證剩下的每筆都高品質。

這個方法已用在[T.4的真實文字實驗](https://birdhackor.github.io/tiny-perceptron-vlm/training.html#T.4)：先合併空白、算完整內容指紋，去掉指紋相同的記錄，再切分。TinyStories的512篇仍是512篇，資料包中的365首詩也仍是365首，這一步沒有再刪掉記錄。原文仍保留自己的空白與換行來訓練；合併空白只是比較內容的規則。注意「沒有找到這種重複」只回答了完整內容是否相同，改寫、同一故事的相似版本還要另查，下一節就看這個缺口。

練習只把第二段改成 `"鳥 看狗"`。先預測正規化不同內容變3、前兩指紋不相同，最後舊assert會失敗，再執行核對。將檢查改成 `hashes[0] != hashes[1]`，確認一個字變化被保留；如果你改的是多一個空格，則本規則仍視為同一內容。
